# Chapter 2: Linear Algebra for Estimation

<a href="../lite/lab/index.html?path=ch02_linear_algebra.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite</a>

*Runs entirely in your browser. No installation required.*

**How to use:** Edit the parameter values in each cell and re-run to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import multivariate_normal

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2.1 Vectors, Matrices, and Geometric Meaning

A robot's state is a vector. A mobile robot on a flat surface has pose $(x, y, \theta)$. A robotic arm might have six joint angles. Every time we write down "the robot is here," we are writing a vector.

Matrices act on vectors. A 2x2 matrix takes every point in the plane and moves it to a new location. The columns of the matrix tell you where the standard basis vectors $\hat{e}_1 = [1, 0]$ and $\hat{e}_2 = [0, 1]$ end up. This completely determines the transformation.

In robotics, the most common matrix operations are rotations (changing coordinate frames) and scaling (converting units or applying gains). Understanding what a matrix does geometrically is essential before we use matrices to represent uncertainty and solve estimation problems.

### Example: What does a matrix do to space?

**Try it:** Change the matrix entries below. Try a rotation (`[[0, -1], [1, 0]]`), a shear (`[[1, 0.5], [0, 1]]`), or a reflection (`[[-1, 0], [0, 1]]`).

```{admonition} What you will build
:class: tip

- Transform a LiDAR scan from sensor coordinates to world coordinates using matrix multiplication
- Fit a line to noisy landmark observations using least squares
- Propagate uncertainty through a nonlinear sensor model using the Jacobian
- Detect outliers using Mahalanobis distance

**Real world application:** These are the mathematical tools behind every SLAM system. After this chapter, you can implement the math that turns raw sensor data into useful robot state estimates.
```

In [ ]:
# PARAMETERS (change these and re-run)
a11, a12 = 1.5, 0.5    # first row of A   (try rotation: 0, -1)
a21, a22 = 0.3, 1.2    # second row of A   (try rotation: 1,  0)

A = np.array([[a11, a12], [a21, a22]])

# Unit circle
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

# Unit square corners
square = np.array([[0,1,1,0,0],[0,0,1,1,0]])

# Apply transformation
ellipse = A @ circle
quad = A @ square

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, orig, transformed, name in [
    (axes[0], circle, ellipse, 'Circle'),
    (axes[1], square, quad, 'Square')
]:
    ax.plot(orig[0], orig[1], 'b-', linewidth=2, label='Original')
    ax.plot(transformed[0], transformed[1], 'r-', linewidth=2, label='Transformed')
    # Show basis vectors and where they go
    ax.annotate('', xy=A[:,0], xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='red', lw=2))
    ax.annotate('', xy=A[:,1], xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
    ax.annotate('', xy=[1,0], xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax.annotate('', xy=[0,1], xytext=[0,0],
                arrowprops=dict(arrowstyle='->', color='darkblue', lw=2))
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f'{name}: Original (blue) vs Transformed (red)')
    ax.legend(fontsize=8)

plt.suptitle(f'A = [[{a11}, {a12}], [{a21}, {a22}]]   det(A) = {np.linalg.det(A):.2f}', fontsize=12)
plt.tight_layout()
plt.show()

### Example: Robot pose as a vector

A mobile robot on a 2D floor has position $(x, y)$. We can visualize this as a point or as an arrow from the origin. The magnitude gives the distance from the origin, and the direction gives the heading.

**Try it:** Move the robot by changing `robot_x` and `robot_y`.

In [ ]:
# PARAMETERS (change these and re-run)
robot_x = 3.0    # robot x position in meters  (try 1, 5, -2)
robot_y = 2.0    # robot y position in meters  (try 0, 4, -3)

magnitude = np.sqrt(robot_x**2 + robot_y**2)
heading = np.degrees(np.arctan2(robot_y, robot_x))

fig, ax = plt.subplots(figsize=(6, 6))
ax.annotate('', xy=[robot_x, robot_y], xytext=[0, 0],
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=3))
ax.plot(robot_x, robot_y, 'o', color='tomato', markersize=12, label='Robot')
ax.plot(0, 0, 's', color='green', markersize=10, label='Origin')

# Draw magnitude arc
arc_theta = np.linspace(0, np.radians(heading) if heading >= 0 else np.radians(heading) + 2*np.pi, 50)
arc_r = 0.8
ax.plot(arc_r * np.cos(arc_theta), arc_r * np.sin(arc_theta), 'g--', alpha=0.5)

ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Robot at ({robot_x}, {robot_y})   |v| = {magnitude:.2f} m   heading = {heading:.1f} deg')
ax.legend()
plt.tight_layout()
plt.show()

**Key observations:**
- The columns of a matrix are the images of the standard basis vectors.
- A matrix is fully determined by what it does to the basis.
- The determinant tells you how much the matrix scales area. If det(A) = 0, the matrix collapses space to a lower dimension.

## 2.2 Matrix Multiplication as Composition

When a robot's lidar reports a point in the sensor frame, we need to transform it to the body frame, then to the world frame. Each of these is a matrix multiplication, and the combined effect is their product.

A critical fact: matrix multiplication is **not commutative**. Rotating then scaling gives a different result than scaling then rotating. In robotics, frame chains are read right to left: $T_{\text{world}} = T_{\text{world} \leftarrow \text{body}} \cdot T_{\text{body} \leftarrow \text{sensor}}$.

### Example: Composing rotation and scaling

**Try it:** Change the rotation angle and scale factor. Toggle `apply_rotation_first` to see that order matters.

In [ ]:
# PARAMETERS (change these and re-run)
rotation_deg = 45.0         # rotation angle in degrees  (try 30, 90, 180)
scale_x = 2.0               # x scaling  (try 0.5, 1.5, 3.0)
scale_y = 0.5               # y scaling  (try 0.5, 1.5, 3.0)
apply_rotation_first = True # True: rotate then scale.  False: scale then rotate.

theta = np.radians(rotation_deg)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.diag([scale_x, scale_y])

if apply_rotation_first:
    combined = S @ R
    title_order = 'Rotate THEN Scale'
else:
    combined = R @ S
    title_order = 'Scale THEN Rotate'

t = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(t), np.sin(t)])
transformed = combined @ circle

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(circle[0], circle[1], 'b-', lw=2)
axes[0].set_title('Original')

if apply_rotation_first:
    step1 = R @ circle
    axes[1].plot(step1[0], step1[1], 'orange', lw=2)
    axes[1].set_title(f'After Rotation ({rotation_deg} deg)')
else:
    step1 = S @ circle
    axes[1].plot(step1[0], step1[1], 'orange', lw=2)
    axes[1].set_title(f'After Scaling ({scale_x}x, {scale_y}x)')

axes[2].plot(transformed[0], transformed[1], 'r-', lw=2)
axes[2].set_title(f'Final: {title_order}')

for ax in axes:
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### Example: Sensor to world frame pipeline

A lidar on a robot detects a wall point at coordinates `(1.5, 0.2)` in the sensor frame. The sensor is mounted at an angle on the robot body, and the robot itself is at some pose in the world. We compose two transformations to get the world coordinates.

**Try it:** Change the sensor mounting angle, robot heading, and robot position.

In [ ]:
# PARAMETERS (change these and re-run)
sensor_point = np.array([1.5, 0.2])   # point in sensor frame (meters)
sensor_mount_deg = 30.0               # sensor mounting angle on body (degrees)
robot_heading_deg = 45.0              # robot heading in world frame (degrees)
robot_pos = np.array([3.0, 2.0])      # robot position in world frame (meters)

def rotation_2d(deg):
    r = np.radians(deg)
    return np.array([[np.cos(r), -np.sin(r)], [np.sin(r), np.cos(r)]])

R_sensor_to_body = rotation_2d(sensor_mount_deg)
R_body_to_world = rotation_2d(robot_heading_deg)

point_body = R_sensor_to_body @ sensor_point
point_world = R_body_to_world @ point_body + robot_pos

fig, ax = plt.subplots(figsize=(8, 8))

# Robot
ax.plot(*robot_pos, 's', color='green', markersize=15, label='Robot', zorder=5)
# Robot heading arrow
heading_vec = rotation_2d(robot_heading_deg) @ np.array([1.0, 0.0])
ax.annotate('', xy=robot_pos + heading_vec, xytext=robot_pos,
            arrowprops=dict(arrowstyle='->', color='green', lw=2))

# Points in each frame
ax.plot(*sensor_point, 'o', color='blue', markersize=10, label=f'Sensor frame {tuple(sensor_point)}')
ax.plot(*point_body, '^', color='orange', markersize=10, label=f'Body frame ({point_body[0]:.2f}, {point_body[1]:.2f})')
ax.plot(*point_world, '*', color='red', markersize=15, label=f'World frame ({point_world[0]:.2f}, {point_world[1]:.2f})')

# Arrows showing the chain
ax.annotate('', xy=point_body, xytext=sensor_point,
            arrowprops=dict(arrowstyle='->', color='orange', lw=1.5, linestyle='--'))
ax.annotate('', xy=point_world, xytext=point_body,
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5, linestyle='--'))

ax.set_xlim(-2, 8); ax.set_ylim(-2, 8)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Sensor to Body to World: frame transformation chain')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

**Key observations:**
- Matrix multiplication is associative but not commutative: $A(BC) = (AB)C$, but $AB \neq BA$ in general.
- Frame chains in robotics are matrix products read right to left.
- Every coordinate transformation in SLAM is a matrix multiplication.

## 2.3 Solving Linear Systems

Many estimation problems reduce to solving $A\mathbf{x} = \mathbf{b}$. When a robot receives range measurements from multiple beacons, it must solve a system of equations to find its position.

Geometrically, each equation is a line (in 2D) or a plane (in 3D). The solution is where they intersect. When we have more equations than unknowns (overdetermined), we use **least squares**: find the $\mathbf{x}$ that minimizes $\|A\mathbf{x} - \mathbf{b}\|^2$.

### Example: Robot trilateration from range beacons

Three beacons at known positions each report a noisy range measurement to the robot. We linearize the range equations and solve for the robot's position.

**Try it:** Change the beacon positions and the noise level.

In [ ]:
# PARAMETERS (change these and re-run)
beacon1 = np.array([0.0, 0.0])    # beacon positions (meters)
beacon2 = np.array([10.0, 0.0])
beacon3 = np.array([5.0, 8.0])
beacon4 = np.array([8.0, 6.0])    # optional 4th beacon
true_pos = np.array([4.0, 3.0])   # true robot position
noise_std = 0.3                    # measurement noise std (meters)  (try 0.0, 0.1, 1.0)
use_4th_beacon = True              # add a 4th beacon for overdetermined system

np.random.seed(42)
beacons = [beacon1, beacon2, beacon3]
if use_4th_beacon:
    beacons.append(beacon4)

# Simulate noisy range measurements
true_ranges = [np.linalg.norm(b - true_pos) for b in beacons]
noisy_ranges = [r + np.random.randn() * noise_std for r in true_ranges]

# Linearize: r_i^2 = (x - bx_i)^2 + (y - by_i)^2
# Subtract last equation from all others to get linear system
n = len(beacons)
A = np.zeros((n - 1, 2))
b = np.zeros(n - 1)
bx_n, by_n = beacons[-1]
r_n = noisy_ranges[-1]
for i in range(n - 1):
    bx_i, by_i = beacons[i]
    r_i = noisy_ranges[i]
    A[i, 0] = 2 * (bx_n - bx_i)
    A[i, 1] = 2 * (by_n - by_i)
    b[i] = r_i**2 - r_n**2 - bx_i**2 - by_i**2 + bx_n**2 + by_n**2

# Solve (least squares if overdetermined)
estimated_pos, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
error = np.linalg.norm(estimated_pos - true_pos)

fig, ax = plt.subplots(figsize=(8, 8))

# Plot beacons and range circles
colors = ['blue', 'green', 'purple', 'cyan']
for i, (bc, r) in enumerate(zip(beacons, noisy_ranges)):
    circle = plt.Circle(bc, r, fill=False, color=colors[i], linestyle='--', alpha=0.5)
    ax.add_patch(circle)
    ax.plot(*bc, 's', color=colors[i], markersize=12)
    ax.annotate(f'B{i+1} (r={r:.1f}m)', bc + 0.2, color=colors[i], fontsize=9)

ax.plot(*true_pos, '*', color='gold', markersize=20, label=f'True pos ({true_pos[0]}, {true_pos[1]})')
ax.plot(*estimated_pos, 'x', color='red', markersize=15, markeredgewidth=3,
        label=f'Estimated ({estimated_pos[0]:.2f}, {estimated_pos[1]:.2f})')

ax.set_xlim(-3, 14); ax.set_ylim(-3, 12)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Trilateration: {n} beacons, noise = {noise_std}m, error = {error:.3f}m')
ax.legend()
plt.tight_layout()
plt.show()

### Example: Line fitting from noisy landmark observations

A robot drives along a corridor and observes the wall at several positions. We fit a line $y = mx + c$ to the noisy measurements using least squares.

**Try it:** Increase the noise or reduce the number of observations.

In [ ]:
# PARAMETERS (change these and re-run)
n_observations = 15     # number of wall observations  (try 3, 5, 30)
wall_slope = 0.5        # true wall slope  (try 0, 1.0, -0.3)
wall_intercept = 2.0    # true wall intercept
noise_level = 0.4       # observation noise std (meters)  (try 0.1, 0.5, 2.0)

np.random.seed(7)
x_obs = np.sort(np.random.uniform(0, 10, n_observations))
y_true = wall_slope * x_obs + wall_intercept
y_obs = y_true + np.random.randn(n_observations) * noise_level

# Set up Ax = b for y = mx + c
A = np.column_stack([x_obs, np.ones(n_observations)])
params, residuals, _, _ = np.linalg.lstsq(A, y_obs, rcond=None)
m_est, c_est = params

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x_obs, y_obs, color='steelblue', s=50, label='Noisy observations', zorder=5)
x_line = np.linspace(0, 10, 100)
ax.plot(x_line, wall_slope * x_line + wall_intercept, 'g--', lw=2, label=f'True wall: y = {wall_slope}x + {wall_intercept}')
ax.plot(x_line, m_est * x_line + c_est, 'r-', lw=2, label=f'Least squares fit: y = {m_est:.2f}x + {c_est:.2f}')

# Draw residuals
for xi, yi in zip(x_obs, y_obs):
    y_fit = m_est * xi + c_est
    ax.plot([xi, xi], [yi, y_fit], 'r-', alpha=0.3, lw=1)

ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Wall fitting: {n_observations} observations, noise = {noise_level}m')
ax.legend()
plt.tight_layout()
plt.show()

**Key observations:**
- When the system is overdetermined (more equations than unknowns), least squares finds the best compromise.
- More measurements reduce the effect of noise on the estimate.
- A singular matrix means the constraints are redundant or contradictory (like parallel lines with no intersection).

## 2.4 Eigenvalues and System Behavior

Eigenvectors are directions that a matrix only stretches (or compresses) without rotating. The eigenvalue is the stretch factor. If $A\mathbf{v} = \lambda \mathbf{v}$, then $\mathbf{v}$ is an eigenvector and $\lambda$ is its eigenvalue.

In estimation, eigenvalues appear in two critical roles:

1. **Covariance interpretation:** The eigenvalues of a covariance matrix are the variances along the principal axes of uncertainty. Large eigenvalue = large uncertainty in that direction.
2. **System stability:** In a discrete dynamical system $\mathbf{x}_{t+1} = A\mathbf{x}_t$, eigenvalues with $|\lambda| < 1$ mean that mode decays over time, while $|\lambda| > 1$ means it grows.

### Example: PCA on lidar scan data

A robot scans a wall with a lidar. The scan produces a cloud of 2D points. Principal Component Analysis (PCA) finds the directions of maximum and minimum variance in this cloud. These directions are exactly the eigenvectors of the data covariance matrix.

This is used in practice for:
- Detecting flat surfaces (walls, floors) in point clouds
- Estimating surface normals for scan matching
- Reducing dimensionality in feature extraction

**Try it:** Change the wall angle and the noise. Notice how the principal components align with and perpendicular to the wall.

In [ ]:
# PARAMETERS (change these and re-run)
wall_angle_deg = 30.0    # wall orientation (degrees)  (try 0, 45, 90)
wall_length = 6.0        # wall length (meters)
n_scan_points = 80       # number of lidar points hitting the wall
along_wall_noise = 0.1   # noise along wall (small, from scan spacing)
perp_wall_noise = 0.15   # noise perpendicular to wall (from range noise)  (try 0.05, 0.3, 0.8)

np.random.seed(10)
wall_angle_rad = np.radians(wall_angle_deg)

# Generate points along the wall with noise
t = np.random.uniform(-wall_length/2, wall_length/2, n_scan_points)
wall_dir = np.array([np.cos(wall_angle_rad), np.sin(wall_angle_rad)])
wall_normal = np.array([-np.sin(wall_angle_rad), np.cos(wall_angle_rad)])

points = (np.outer(t, wall_dir)
          + np.outer(np.random.randn(n_scan_points) * along_wall_noise, wall_dir)
          + np.outer(np.random.randn(n_scan_points) * perp_wall_noise, wall_normal))

# Shift wall to a position
wall_center = np.array([4.0, 3.0])
points += wall_center

# PCA: compute covariance and eigenvectors
mean = points.mean(axis=0)
centered = points - mean
cov_matrix = (centered.T @ centered) / (n_scan_points - 1)
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort by descending eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: point cloud with principal components
ax = axes[0]
ax.scatter(points[:, 0], points[:, 1], c='steelblue', s=15, alpha=0.6, label='Lidar points')
ax.plot(*mean, 'r+', markersize=15, markeredgewidth=3, label='Mean')

colors = ['red', 'orange']
labels = ['PC1 (max variance)', 'PC2 (min variance)']
for i in range(2):
    scale = 2 * np.sqrt(eigenvalues[i])  # 2 sigma length
    direction = eigenvectors[:, i] * scale
    ax.annotate('', xy=mean + direction, xytext=mean,
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=3))
    ax.annotate(labels[i], xy=mean + direction * 1.1, color=colors[i], fontsize=9, fontweight='bold')

# Draw covariance ellipse
angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
for ns in [1, 2]:
    ell = patches.Ellipse(mean, 2*ns*np.sqrt(eigenvalues[0]), 2*ns*np.sqrt(eigenvalues[1]),
                          angle=angle, fill=False, edgecolor='tomato', linewidth=1.5, linestyle=':')
    ax.add_patch(ell)

ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'PCA on Lidar Wall Scan (wall at {wall_angle_deg} deg)')
ax.legend(fontsize=8)

# Right: eigenvalue bar chart
ax2 = axes[1]
ax2.bar(['PC1 (along wall)', 'PC2 (perpendicular)'], eigenvalues, color=['red', 'orange'])
ax2.set_ylabel('Eigenvalue (variance)')
ax2.set_title('Eigenvalue Magnitudes')
for i, v in enumerate(eigenvalues):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

ratio = eigenvalues[0] / eigenvalues[1] if eigenvalues[1] > 1e-10 else float('inf')
ax2.text(0.5, 0.85, f'Ratio PC1/PC2 = {ratio:.1f}\n(high ratio = flat surface)',
         transform=ax2.transAxes, ha='center', fontsize=11,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

print(f'Covariance matrix:')
print(f'  [[{cov_matrix[0,0]:.4f}, {cov_matrix[0,1]:.4f}]')
print(f'   [{cov_matrix[1,0]:.4f}, {cov_matrix[1,1]:.4f}]]')
print(f'\nEigenvector 1 (along wall):      [{eigenvectors[0,0]:.3f}, {eigenvectors[1,0]:.3f}]')
print(f'Eigenvector 2 (wall normal):      [{eigenvectors[0,1]:.3f}, {eigenvectors[1,1]:.3f}]')
print(f'True wall direction:              [{wall_dir[0]:.3f}, {wall_dir[1]:.3f}]')

### Example: Repeated matrix application and stability

In a discrete motion model $\mathbf{x}_{t+1} = A\mathbf{x}_t$, applying the matrix repeatedly reveals whether errors grow or shrink. This depends entirely on the eigenvalues of $A$.

Imagine a robot's position error propagating through a motion model over many timesteps. If any eigenvalue has magnitude greater than 1, that error component grows exponentially.

**Try it:** Change the matrix entries. Stable systems have all eigenvalue magnitudes below 1.

In [ ]:
# PARAMETERS (change these and re-run)
# Motion model matrix (2x2)
# Stable example: [[0.9, 0.1], [-0.1, 0.8]]
# Unstable example: [[1.05, 0.1], [0.0, 0.95]]
A = np.array([[0.9, 0.1],
              [-0.1, 0.8]])
x0 = np.array([1.0, 0.5])   # initial error vector
n_steps = 40                  # number of timesteps

eigenvalues_A = np.linalg.eigvals(A)
max_eig_mag = max(abs(eigenvalues_A))

trajectory = [x0.copy()]
x = x0.copy()
for _ in range(n_steps):
    x = A @ x
    trajectory.append(x.copy())
trajectory = np.array(trajectory)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: trajectory in state space
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0, 1, len(trajectory)))
ax.scatter(trajectory[:, 0], trajectory[:, 1], c=colors, s=30, zorder=5)
ax.plot(trajectory[:, 0], trajectory[:, 1], 'k-', alpha=0.3)
ax.plot(*x0, 'go', markersize=12, label='Start')
ax.plot(*trajectory[-1], 'rx', markersize=12, markeredgewidth=3, label='End')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
stability = 'STABLE' if max_eig_mag < 1 else 'UNSTABLE'
ax.set_title(f'Error trajectory ({stability})')
ax.legend()

# Right: magnitude over time
ax2 = axes[1]
magnitudes = np.linalg.norm(trajectory, axis=1)
ax2.plot(magnitudes, 'steelblue', lw=2)
ax2.axhline(magnitudes[0], color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Timestep'); ax2.set_ylabel('Error magnitude')
ax2.set_title(f'Eigenvalues: {eigenvalues_A[0]:.3f}, {eigenvalues_A[1]:.3f}  (max |λ| = {max_eig_mag:.3f})')

plt.tight_layout()
plt.show()

**Key observations:**
- Eigenvalues of a symmetric matrix are always real.
- The eigenvectors of a covariance matrix give the principal axes of uncertainty (PCA).
- A high ratio between the largest and smallest eigenvalue means the data or uncertainty is highly directional.
- In dynamical systems, all eigenvalue magnitudes must be below 1 for stability.

## 2.5 Positive Definite Matrices and Covariance

A symmetric matrix is **positive definite** (PD) if all its eigenvalues are positive. Geometrically, the quadratic form $\mathbf{x}^\top A \mathbf{x} = 1$ traces an ellipse when $A$ is PD, a hyperbola when $A$ is indefinite, and nothing when $A$ is negative definite.

Covariance matrices must be positive semi-definite. If a covariance matrix has a negative eigenvalue, the "uncertainty" it describes is physically meaningless. This is why numerical algorithms in SLAM are carefully designed to preserve positive definiteness.

The **Cholesky decomposition** factors $\Sigma = LL^\top$ where $L$ is lower triangular. It is the matrix equivalent of taking a square root and is used extensively in sampling, solving systems, and maintaining numerical stability.

### Example: Is this covariance matrix valid?

**Try it:** Enter a 2x2 symmetric matrix and check whether it is a valid covariance matrix.

In [ ]:
# PARAMETERS (change these and re-run)
# Symmetric matrix: [[a, b], [b, d]]
a = 4.0    # variance in x  (try 4.0, 1.0, -1.0)
b = 1.5    # covariance     (try 0.0, 1.5, 5.0)
d = 1.0    # variance in y  (try 1.0, 0.5, 2.0)

M = np.array([[a, b], [b, d]])
evals = np.linalg.eigvalsh(M)
det = np.linalg.det(M)
trace = np.trace(M)
is_pd = np.all(evals > 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: quadratic form visualization
ax = axes[0]
x = np.linspace(-4, 4, 300)
y = np.linspace(-4, 4, 300)
X, Y = np.meshgrid(x, y)
Z = a * X**2 + 2 * b * X * Y + d * Y**2
ax.contour(X, Y, Z, levels=[1, 2, 4, 8], colors='steelblue', linewidths=2)
ax.contourf(X, Y, Z, levels=20, cmap='Blues', alpha=0.3)
ax.set_aspect('equal')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
status = 'VALID (Positive Definite)' if is_pd else 'INVALID (Not Positive Definite)'
color = 'green' if is_pd else 'red'
ax.set_title(f'Quadratic form $x^T M x$ = const', color=color)

# Right: diagnostics
ax2 = axes[1]; ax2.axis('off')
info = (
    f'Matrix M:\n'
    f'  [[{a:.2f}, {b:.2f}]\n'
    f'   [{b:.2f}, {d:.2f}]]\n\n'
    f'Eigenvalues: {evals[0]:.3f}, {evals[1]:.3f}\n'
    f'Determinant: {det:.3f}\n'
    f'Trace: {trace:.3f}\n\n'
    f'Status: {status}'
)
ax2.text(0.1, 0.5, info, transform=ax2.transAxes, fontsize=13,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightgreen' if is_pd else 'lightyellow'))

plt.tight_layout()
plt.show()

### Example: Sampling from a covariance using Cholesky

To simulate noisy robot poses, we need to sample from a Gaussian with a given covariance. The Cholesky factor $L$ transforms standard normal samples into correlated samples: $\mathbf{x} = L \mathbf{z}$ where $\mathbf{z} \sim \mathcal{N}(0, I)$.

**Try it:** Change the covariance parameters and see how the sample cloud matches the ellipse.

In [ ]:
# PARAMETERS (change these and re-run)
sigma_x = 2.0    # std dev in x (meters)  (try 0.5, 2.0, 4.0)
sigma_y = 0.8    # std dev in y (meters)  (try 0.3, 1.0, 3.0)
rho = 0.6        # correlation  (try -0.9, 0.0, 0.6)
n_samples = 500  # number of samples  (try 50, 500, 2000)

rho = np.clip(rho, -0.99, 0.99)
Sigma = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                  [rho*sigma_x*sigma_y, sigma_y**2]])

# Cholesky decomposition
L = np.linalg.cholesky(Sigma)

# Sample: x = L @ z where z ~ N(0, I)
np.random.seed(42)
z = np.random.randn(2, n_samples)
samples = (L @ z).T

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(samples[:, 0], samples[:, 1], s=5, alpha=0.4, color='steelblue', label='Samples')

# Draw covariance ellipses
evals, evecs = np.linalg.eigh(Sigma)
angle = np.degrees(np.arctan2(evecs[1, 1], evecs[0, 1]))
for ns, c, lbl in [(1, 'tomato', '1 sigma'), (2, 'orange', '2 sigma'), (3, 'gold', '3 sigma')]:
    ell = patches.Ellipse((0, 0), 2*ns*np.sqrt(evals[1]), 2*ns*np.sqrt(evals[0]),
                          angle=angle, fill=False, edgecolor=c, linewidth=2, label=lbl)
    ax.add_patch(ell)

ax.set_xlim(-8, 8); ax.set_ylim(-8, 8)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Sampling via Cholesky: {n_samples} samples, rho = {rho}')
ax.legend()

# Count samples inside each ellipse
Sigma_inv = np.linalg.inv(Sigma)
mahal = np.array([s @ Sigma_inv @ s for s in samples])
for ns in [1, 2, 3]:
    pct = 100 * np.mean(mahal <= ns**2)
    print(f'  Points inside {ns} sigma ellipse: {pct:.1f}% (expected: {[68.3, 95.4, 99.7][ns-1]}%)')

plt.tight_layout()
plt.show()

print(f'\nCholesky factor L:')
print(f'  [[{L[0,0]:.3f}, {L[0,1]:.3f}]')
print(f'   [{L[1,0]:.3f}, {L[1,1]:.3f}]]')

**Key observations:**
- A covariance matrix is always symmetric positive semi-definite.
- If you build a covariance matrix and it is not PD, something is wrong with your model.
- Cholesky decomposition is numerically stable and faster than eigendecomposition for sampling.

## 2.6 Quadratic Forms and Error Representation

The expression $(\mathbf{x} - \boldsymbol{\mu})^\top \boldsymbol{\Sigma}^{-1} (\mathbf{x} - \boldsymbol{\mu})$ is the **Mahalanobis distance**. It measures how far $\mathbf{x}$ is from $\boldsymbol{\mu}$, accounting for the shape of the uncertainty.

This quadratic form appears everywhere in estimation:
- In the exponent of the Gaussian distribution
- In weighted least squares cost functions
- In the Kalman filter innovation test

The Euclidean distance treats all directions equally. The Mahalanobis distance uses the covariance to weight directions. A measurement that falls along the high variance axis is less surprising than one that falls along the low variance axis.

### Example: Euclidean vs Mahalanobis distance

A robot believes it is at position $\boldsymbol{\mu}$ with covariance $\boldsymbol{\Sigma}$. It receives a GPS fix at a different location. Is this fix consistent with the robot's belief? The Mahalanobis distance tells us.

**Try it:** Move the GPS fix and see how the two distances differ.

In [ ]:
# PARAMETERS (change these and re-run)
mu = np.array([0.0, 0.0])         # robot's believed position
sigma_x = 3.0                      # uncertainty in x (large: along corridor)
sigma_y = 0.5                      # uncertainty in y (small: corridor width)
rho = 0.0                          # correlation
gps_fix = np.array([2.5, 0.8])    # GPS measurement  (try [2.5, 0.8], [0.5, 2.0], [5.0, 0.0])

Sigma = np.array([[sigma_x**2, rho*sigma_x*sigma_y],
                  [rho*sigma_x*sigma_y, sigma_y**2]])
Sigma_inv = np.linalg.inv(Sigma)

diff = gps_fix - mu
euclidean_dist = np.linalg.norm(diff)
mahal_dist = np.sqrt(diff @ Sigma_inv @ diff)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, dist_type in zip(axes, ['euclidean', 'mahalanobis']):
    # Draw contours
    if dist_type == 'euclidean':
        for r in [1, 2, 3, 4]:
            circle = plt.Circle(mu, r, fill=False, color='gray', linestyle=':', alpha=0.5)
            ax.add_patch(circle)
        ax.set_title(f'Euclidean distance = {euclidean_dist:.2f}m')
    else:
        evals, evecs = np.linalg.eigh(Sigma)
        angle = np.degrees(np.arctan2(evecs[1, 1], evecs[0, 1]))
        for ns in [1, 2, 3, 4]:
            ell = patches.Ellipse(mu, 2*ns*np.sqrt(evals[1]), 2*ns*np.sqrt(evals[0]),
                                  angle=angle, fill=False, color='steelblue', linestyle=':', alpha=0.5)
            ax.add_patch(ell)
        ax.set_title(f'Mahalanobis distance = {mahal_dist:.2f} sigma')

    ax.plot(*mu, 'go', markersize=12, label='Robot belief')
    ax.plot(*gps_fix, 'r*', markersize=15, label='GPS fix')
    ax.plot([mu[0], gps_fix[0]], [mu[1], gps_fix[1]], 'r--', lw=1.5)
    ax.set_xlim(-8, 8); ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

print(f'Euclidean distance:  {euclidean_dist:.2f} m')
print(f'Mahalanobis distance: {mahal_dist:.2f} sigma')
if mahal_dist < 2:
    print('GPS fix is consistent with the robot belief (within 2 sigma).')
else:
    print('GPS fix is suspicious! It falls outside 2 sigma.')

### Example: Weighted least squares cost surface

A robot estimates its 2D position from three range beacons, each with a different noise level. The cost function is a sum of weighted squared residuals. The shape of the cost surface reveals how the sensor geometry affects the estimate.

**Try it:** Change the noise levels of individual sensors to see how the cost surface and the estimate shift.

In [ ]:
# PARAMETERS (change these and re-run)
b1, sigma1 = np.array([0.0, 0.0]), 0.5    # beacon 1: position and noise std
b2, sigma2 = np.array([8.0, 0.0]), 1.5    # beacon 2: noisier sensor
b3, sigma3 = np.array([4.0, 7.0]), 0.3    # beacon 3: very precise sensor
true_pos = np.array([3.5, 2.5])

np.random.seed(99)
beacons = [(b1, sigma1), (b2, sigma2), (b3, sigma3)]
measured_ranges = [np.linalg.norm(b - true_pos) + np.random.randn() * s for b, s in beacons]

# Cost function over a grid
xg = np.linspace(-1, 10, 200)
yg = np.linspace(-1, 10, 200)
Xg, Yg = np.meshgrid(xg, yg)
cost = np.zeros_like(Xg)

for (b, sigma), r_meas in zip(beacons, measured_ranges):
    r_pred = np.sqrt((Xg - b[0])**2 + (Yg - b[1])**2)
    cost += ((r_pred - r_meas) / sigma)**2

# Find minimum
min_idx = np.unravel_index(np.argmin(cost), cost.shape)
est_pos = np.array([Xg[min_idx], Yg[min_idx]])

fig, ax = plt.subplots(figsize=(8, 8))
cs = ax.contourf(Xg, Yg, np.log(cost + 1), levels=30, cmap='RdYlBu_r', alpha=0.7)
ax.contour(Xg, Yg, cost, levels=[1, 4, 9, 16, 25], colors='white', linewidths=0.8)

colors = ['blue', 'green', 'purple']
for i, ((b, sigma), r) in enumerate(zip(beacons, measured_ranges)):
    ax.plot(*b, 's', color=colors[i], markersize=12)
    ax.annotate(f'B{i+1} (sigma={sigma})', b + 0.2, color=colors[i], fontweight='bold')

ax.plot(*true_pos, '*', color='gold', markersize=18, label=f'True: ({true_pos[0]}, {true_pos[1]})')
ax.plot(*est_pos, 'rx', markersize=15, markeredgewidth=3, label=f'Estimate: ({est_pos[0]:.1f}, {est_pos[1]:.1f})')

ax.set_xlim(-1, 10); ax.set_ylim(-1, 10)
ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Weighted Least Squares Cost Surface')
ax.legend(loc='upper right')
plt.colorbar(cs, ax=ax, label='log(cost)')
plt.tight_layout()
plt.show()

**Key observations:**
- Mahalanobis distance is the natural "ruler" in estimation problems. It accounts for the shape of uncertainty.
- Minimizing a sum of weighted squared residuals is equivalent to finding the maximum likelihood estimate under Gaussian noise.
- Sensors with lower noise (smaller sigma) pull the estimate more strongly toward their measurement.

## 2.7 Jacobians as Local Linear Models

Most sensor and motion models in robotics are nonlinear. A lidar returns range and bearing $(r, \phi)$, but we need Cartesian coordinates $(x, y)$. The relationship $x = r\cos\phi$, $y = r\sin\phi$ is nonlinear.

The **Jacobian** is the matrix of partial derivatives that gives the best linear approximation of a nonlinear function at a specific point. It is the foundation of the Extended Kalman Filter (EKF), which we will study in Chapter 17.

The Jacobian also tells us how uncertainty transforms through nonlinear functions. If $\mathbf{y} = f(\mathbf{x})$ and $\mathbf{x}$ has covariance $\boldsymbol{\Sigma}_x$, then:

$$\boldsymbol{\Sigma}_y \approx J \boldsymbol{\Sigma}_x J^\top$$

where $J$ is the Jacobian of $f$ evaluated at the mean of $\mathbf{x}$. This is the most important formula for uncertainty propagation.

### Example: Linearizing polar to Cartesian (lidar model)

A lidar measurement $(r, \theta)$ converts to Cartesian as $f(r, \theta) = (r\cos\theta, r\sin\theta)$. The Jacobian at a point $(r_0, \theta_0)$ is:

$$J = \begin{bmatrix} \cos\theta_0 & -r_0\sin\theta_0 \\ \sin\theta_0 & r_0\cos\theta_0 \end{bmatrix}$$

**Try it:** Change the linearization point. Notice how the linear approximation degrades far from the point.

In [ ]:
# PARAMETERS (change these and re-run)
r0 = 5.0            # range at linearization point (meters)  (try 2, 5, 10)
theta0_deg = 45.0    # bearing at linearization point (degrees)  (try 0, 45, 90, 135)

theta0 = np.radians(theta0_deg)

# Nonlinear function: polar to Cartesian
def polar_to_cart(r, theta):
    return np.array([r * np.cos(theta), r * np.sin(theta)])

# Jacobian at (r0, theta0)
J = np.array([[np.cos(theta0), -r0 * np.sin(theta0)],
              [np.sin(theta0),  r0 * np.cos(theta0)]])

# Generate a grid of points in polar space around (r0, theta0)
dr = np.linspace(-2, 2, 15)
dtheta = np.linspace(-0.4, 0.4, 15)
DR, DTHETA = np.meshgrid(dr, dtheta)

# True nonlinear mapping
cart_true = np.array([polar_to_cart(r0 + dr_i, theta0 + dt_i)
                      for dr_i, dt_i in zip(DR.ravel(), DTHETA.ravel())])

# Linear approximation: f(r0,theta0) + J @ [dr, dtheta]
f0 = polar_to_cart(r0, theta0)
cart_linear = np.array([f0 + J @ np.array([dr_i, dt_i])
                        for dr_i, dt_i in zip(DR.ravel(), DTHETA.ravel())])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: polar grid
ax = axes[0]
ax.scatter(r0 + DR.ravel(), np.degrees(theta0 + DTHETA.ravel()), c='steelblue', s=10)
ax.plot(r0, theta0_deg, 'r*', markersize=15, label='Linearization point')
ax.set_xlabel('Range (m)'); ax.set_ylabel('Bearing (deg)')
ax.set_title('Polar (input) space')
ax.legend()

# Right: Cartesian outputs
ax2 = axes[1]
ax2.scatter(cart_true[:, 0], cart_true[:, 1], c='steelblue', s=15, alpha=0.7, label='True (nonlinear)')
ax2.scatter(cart_linear[:, 0], cart_linear[:, 1], c='tomato', s=15, alpha=0.7, label='Linear approx (Jacobian)')
ax2.plot(*f0, 'r*', markersize=15)
ax2.set_xlabel('x (m)'); ax2.set_ylabel('y (m)')
ax2.set_title('Cartesian (output) space')
ax2.set_aspect('equal')
ax2.legend()

plt.tight_layout()
plt.show()

### Example: Uncertainty propagation through a nonlinear sensor model

A lidar measurement has uncertainty in both range and bearing. When we convert to Cartesian, the Gaussian in polar space becomes a "banana" shape. The Jacobian approximation captures this as an ellipse, which works well when the uncertainty is small.

**Try it:** Increase `sigma_theta` to see when the linear approximation breaks down.

In [ ]:
# PARAMETERS (change these and re-run)
r0 = 5.0                  # mean range (meters)
theta0_deg = 60.0          # mean bearing (degrees)
sigma_r = 0.3              # range uncertainty (meters)  (try 0.1, 0.3, 1.0)
sigma_theta_deg = 5.0      # bearing uncertainty (degrees)  (try 2, 5, 15, 30)
n_samples = 1000

theta0 = np.radians(theta0_deg)
sigma_theta = np.radians(sigma_theta_deg)

# Covariance in polar space
Sigma_polar = np.array([[sigma_r**2, 0],
                         [0, sigma_theta**2]])

# Jacobian at (r0, theta0)
J = np.array([[np.cos(theta0), -r0 * np.sin(theta0)],
              [np.sin(theta0),  r0 * np.cos(theta0)]])

# Propagated covariance (linear approximation)
Sigma_cart = J @ Sigma_polar @ J.T

# Monte Carlo: sample polar, transform nonlinearly
np.random.seed(42)
polar_samples = np.random.multivariate_normal([r0, theta0], Sigma_polar, n_samples)
cart_samples = np.column_stack([polar_samples[:, 0] * np.cos(polar_samples[:, 1]),
                                 polar_samples[:, 0] * np.sin(polar_samples[:, 1])])

# Mean in Cartesian
f0 = polar_to_cart(r0, theta0)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(cart_samples[:, 0], cart_samples[:, 1], s=3, alpha=0.3, color='steelblue',
           label='Monte Carlo samples')

# Draw linearized covariance ellipses
evals, evecs = np.linalg.eigh(Sigma_cart)
angle = np.degrees(np.arctan2(evecs[1, 1], evecs[0, 1]))
for ns, c, lbl in [(1, 'tomato', '1 sigma (Jacobian)'), (2, 'orange', '2 sigma (Jacobian)')]:
    ell = patches.Ellipse(f0, 2*ns*np.sqrt(evals[1]), 2*ns*np.sqrt(evals[0]),
                          angle=angle, fill=False, edgecolor=c, linewidth=2.5, label=lbl)
    ax.add_patch(ell)

ax.plot(*f0, 'r*', markersize=15, label='Mean')
ax.plot(0, 0, 'gs', markersize=10, label='Robot (sensor origin)')

ax.set_aspect('equal')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title(f'Uncertainty propagation: sigma_r={sigma_r}m, sigma_theta={sigma_theta_deg} deg')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

### Example: Numerical vs analytical Jacobian

Computing Jacobians by hand is tedious and error prone. In practice, you can verify your analytical Jacobian against a numerical one computed via finite differences.

**Try it:** Change the evaluation point and the step size `eps`.

In [ ]:
# PARAMETERS (change these and re-run)
r_eval = 5.0           # range at evaluation point (meters)
theta_eval_deg = 60.0  # bearing at evaluation point (degrees)
eps = 1e-6             # finite difference step size  (try 1e-4, 1e-6, 1e-8)

theta_eval = np.radians(theta_eval_deg)
point = np.array([r_eval, theta_eval])

# Analytical Jacobian
J_analytical = np.array([[np.cos(theta_eval), -r_eval * np.sin(theta_eval)],
                          [np.sin(theta_eval),  r_eval * np.cos(theta_eval)]])

# Numerical Jacobian via central differences
def f(p):
    return np.array([p[0] * np.cos(p[1]), p[0] * np.sin(p[1])])

J_numerical = np.zeros((2, 2))
for j in range(2):
    e = np.zeros(2)
    e[j] = eps
    J_numerical[:, j] = (f(point + e) - f(point - e)) / (2 * eps)

print('Analytical Jacobian:')
print(f'  [[{J_analytical[0,0]:.6f}, {J_analytical[0,1]:.6f}]')
print(f'   [{J_analytical[1,0]:.6f}, {J_analytical[1,1]:.6f}]]')
print()
print('Numerical Jacobian:')
print(f'  [[{J_numerical[0,0]:.6f}, {J_numerical[0,1]:.6f}]')
print(f'   [{J_numerical[1,0]:.6f}, {J_numerical[1,1]:.6f}]]')
print()
max_error = np.max(np.abs(J_analytical - J_numerical))
print(f'Max element-wise error: {max_error:.2e}')
print(f'Step size used: {eps:.1e}')

**Key observations:**
- The Jacobian is the bridge between nonlinear reality and linear algebra tools.
- $\boldsymbol{\Sigma}_y = J \boldsymbol{\Sigma}_x J^\top$ is the single most important formula for uncertainty propagation.
- When the uncertainty is large relative to the curvature of the function, the linearization breaks down and the ellipse approximation becomes poor.
- Numerical Jacobians are a useful debugging tool even when analytical ones are available.

## Exercises

### Exercise 2.1: Rotation and magnitude preservation

Build a rotation matrix for 60 degrees. Apply it to the vector $[3, 1]$. Plot both the original and rotated vectors. Verify that the magnitude is preserved (the lengths should be identical).

In [ ]:
# Your code here
angle_deg = 60.0
v = np.array([3, 1])

# Build rotation matrix R
R = ...  # fill this in

# Apply rotation
v_rotated = ...  # fill this in

# Print magnitudes
print(f'Original magnitude: ...')
print(f'Rotated magnitude: ...')

# Plot both vectors as arrows from origin
# ...

### Exercise 2.2: Beacon localization

A robot receives range measurements from three beacons:
- Beacon at (0, 0): measured range = 5.0 m
- Beacon at (10, 0): measured range = 4.2 m
- Beacon at (5, 8): measured range = 6.1 m

Set up the linearized system and solve for the robot's position using `np.linalg.lstsq`. Plot the beacons, range circles, and your estimated position.

In [ ]:
# Your code here
beacons = [np.array([0, 0]), np.array([10, 0]), np.array([5, 8])]
ranges = [5.0, 4.2, 6.1]

# Linearize the range equations (subtract the last equation from the others)
# Solve for [x, y]
# Plot the result
# ...

### Exercise 2.3: PCA on a point cloud

Generate a set of 100 points sampled from a 2D Gaussian with mean $[5, 3]$ and covariance $\Sigma = [[4, 1.5], [1.5, 1]]$. Compute the sample covariance, find its eigenvalues and eigenvectors, and draw the 1 sigma and 2 sigma ellipses. Verify the ellipse axes match the eigenvectors.

In [ ]:
# Your code here
mu = np.array([5, 3])
Sigma = np.array([[4, 1.5], [1.5, 1]])

# Sample 100 points
# Compute sample covariance
# Find eigenvalues and eigenvectors
# Plot points, eigenvectors, and ellipses
# ...

### Exercise 2.4: Mahalanobis distance for outlier detection

A robot has pose estimate $\boldsymbol{\mu} = [1, 1]$ with covariance $\boldsymbol{\Sigma} = [[2, 0.5], [0.5, 1]]$. It receives five GPS measurements:

- A = (1.5, 1.2)
- B = (3.0, 2.0)
- C = (0.5, 4.0)
- D = (2.0, 0.5)
- E = (5.0, 5.0)

Compute the Mahalanobis distance for each. Which measurements are consistent (within 2 sigma)? Which are likely outliers?

In [ ]:
# Your code here
mu = np.array([1, 1])
Sigma = np.array([[2, 0.5], [0.5, 1]])
measurements = {'A': [1.5, 1.2], 'B': [3.0, 2.0], 'C': [0.5, 4.0], 'D': [2.0, 0.5], 'E': [5.0, 5.0]}

# Compute Mahalanobis distance for each
# Classify as inlier (< 2 sigma) or outlier (>= 2 sigma)
# ...

### Exercise 2.5: Observation model Jacobian (challenge)

A robot at pose $(x, y, \theta)$ observes a landmark at $(l_x, l_y)$. The observation model gives range and bearing:

$$r = \sqrt{(l_x - x)^2 + (l_y - y)^2}$$
$$\phi = \text{atan2}(l_y - y,\  l_x - x) - \theta$$

Compute the Jacobian of $[r, \phi]$ with respect to $[x, y, \theta]$ analytically. Verify your result against a numerical Jacobian at the point $x=1, y=2, \theta=0.3$ with a landmark at $l_x=5, l_y=4$.

In [ ]:
# Your code here
x, y, theta = 1.0, 2.0, 0.3
lx, ly = 5.0, 4.0

# Compute analytical Jacobian (2x3 matrix)
# J = [[dr/dx, dr/dy, dr/dtheta],
#      [dphi/dx, dphi/dy, dphi/dtheta]]

# Compute numerical Jacobian using finite differences

# Compare the two
# ...

### Exercise 2.6: Uncertainty propagation through observation model (challenge)

Using the Jacobian from Exercise 2.5, propagate the robot's pose covariance through the observation model to get the observation covariance. Use:

$$\boldsymbol{\Sigma}_{\text{pose}} = \begin{bmatrix} 0.1 & 0.02 & 0.01 \\ 0.02 & 0.15 & 0.0 \\ 0.01 & 0.0 & 0.05 \end{bmatrix}$$

Compute $\boldsymbol{\Sigma}_{\text{obs}} = J \boldsymbol{\Sigma}_{\text{pose}} J^\top$. Verify by Monte Carlo: sample 1000 poses, transform each through the observation model, and scatter plot the results. Overlay the linearized 2 sigma ellipse.

In [ ]:
# Your code here
Sigma_pose = np.array([[0.1, 0.02, 0.01],
                        [0.02, 0.15, 0.0],
                        [0.01, 0.0, 0.05]])

# Use J from Exercise 2.5
# Compute Sigma_obs = J @ Sigma_pose @ J.T
# Monte Carlo verification: sample poses, compute observations, scatter plot
# ...